So, this one is the exact opposite of the main application:

- Everyhing here is writted 100% manually. I might have asked "Chatgpt" abstract questions, but it wrote no code at all.
- (almost) No libraries. The code rawdogs the LLM calls with pure REST.

I tried it with `Qwen3.6-9B` and `gpt-oss-20b` locally, but both are not smart enough without reasoning and too slow with reasoning. Larger models are also too slow on my HW.
So I ran it with `Kimi K 2.6` on fireworks.ai - it seems to be smart enough to not make any mistakes, and also decently fast. A good model actually.


In [ ]:
from dataclasses import dataclass, field
import json
import aiohttp
from typing import Optional, Literal, Callable, Annotated
import os
from dotenv import load_dotenv

load_dotenv()

True

Let's parse ser-des the messages we get from the api to/frop dataclasses.


In [ ]:
# Assistant message may contain 1 or more toolcall, which will be have type="function", and the funciton itself is of type FunctionCall
@dataclass
class FunctionCall:
    name: str
    arguments: dict

@dataclass
class ToolCall:
    id: str
    function: FunctionCall
    type: Literal["function"] = "function"

# Let's not over-complicate things - one class for all message types.
@dataclass
class Message:
    role: Literal["system", "user", "assistant", "tool"]
    content: str | None = None
    tool_calls: list[ToolCall] = field(default_factory=list)  # For assistant messages, optional tool calls.
    tool_call_id: str | None = None  # For "tool" message - the tool call it for the tool call result.

def parse_toolcall(t: dict):
    function = t["function"]
    return ToolCall(id=t["id"], type="function", function=FunctionCall(name=function["name"], arguments=json.loads(function["arguments"])))

def serialize_toolcall(t: ToolCall):
    return {"id": t.id, "type": t.type, "function": {"name": t.function.name, "arguments": json.dumps(t.function.arguments)}}

def parse_message(r: dict) -> Message:
    choices = r["choices"]
    message = choices[0]["message"]

    mm = Message(role=message["role"], content=message["content"])
    if "tool_calls" in message:
        tc = message["tool_calls"]
        mm.tool_calls = [parse_toolcall(t) for t in tc]

    return mm

def serialize_message(m: Message) -> dict:
    res = {"role": m.role, "content": m.content}

    if m.role == "assistant" and m.tool_calls:
        res |= {"tool_calls": [serialize_toolcall(t) for t in m.tool_calls]}

    if m.role == "tool":
        res |= {"tool_call_id": m.tool_call_id}

    return res


This is how we tell the llm which tools are available


In [ ]:
from typing import get_type_hints, get_origin, get_args

# Convert an annotated python function into the schema for chat completion API.

def tool_schema(f: Callable):
    type_map = {"str": "string", "int": "integer", "bool": "boolean"}
    hints = get_type_hints(f, include_extras=True)
    args = {}

    for name, ann in hints.items():
        if name == 'return':
            continue  # This is for the return type (reserved keyword anyway)
        if get_origin(ann) is Annotated:
            meta = get_args(ann)
            args[name] = (meta[0].__name__, meta[1])
        else:
            args[name] = (ann.__name__, "")

    return {
        "type": "function",
        "function": {
            "name": f.__name__,
            "description": f.__doc__,
            "parameters": {
                "type": "object",
                "properties": {
                    k: {
                        "type": type_map[v[0]],
                        "description": v[1]
                    }
                    for k, v in args.items()
                },
                "required": list(args.keys())  # Assume all args are required for simplicity
            },
        }
    }

In [5]:
import json

def test(a: Annotated[str, "first"], b: int):
    """asdf sdfs sfsf"""

print(json.dumps(
    tool_schema(test),
    indent=2,
))

{
  "type": "function",
  "function": {
    "name": "test",
    "description": "asdf sdfs sfsf",
    "parameters": {
      "type": "object",
      "properties": {
        "a": {
          "type": "string",
          "description": "first"
        },
        "b": {
          "type": "integer",
          "description": ""
        }
      },
      "required": [
        "a",
        "b"
      ]
    }
  }
}


Now, let's work with the model.


In [43]:
from pprint import pp

class Model:

    id: str
    api_key: Optional[str]
    base_url: str

    def __init__(self, id: str, api_key: str | None = None, base_url="http://127.0.0.1:3456/v1/"):
        self.id = id
        self.api_key = api_key
        self.session = aiohttp.ClientSession(
            base_url, headers={"Content-Type": "application/json"} | ({
                "Authorization": f"Bearer {api_key}"
            } if api_key else {})
        )

    # We want to close the session, but we can't close it in __del__, and it's not even the right way.
    # So let's use the model as an async context manager instead.
    async def __aenter__(self):
        return self

    async def __aexit__(self, exc_type, exc, tb):
        if self.session: await self.session.close()

    async def generate(
        self,
        history: list[Message] | None = None,
        tools=(),
        tool_choice: Literal["required", "auto", "none"] = "auto",
        reasoning_effort: str = "none"
    ):
        history = [] if history is None else history
        tools_schemas = [tool_schema(t) for t in tools]

        if tool_choice == "required": assert tools_schemas

        async with self.session.post(
            "chat/completions",
            json=(
                j := {
                    "model": self.id,
                    "messages": [serialize_message(m) for m in history],
                    "tools": tools_schemas,
                    "tool_choice": tool_choice,
                    "reasoning_effort": reasoning_effort
                }
            )
        ) as r:
            # print(pp(j))
            # print(await r.json())
            r.raise_for_status()
            return parse_message(await r.json())


Let's test it with tool calls


In [47]:
def get_weather(location: Annotated[str, "Where?"]):
    """get the effing weather"""

    print(f"get_weather({location})")

    if "hell" in location.lower(): return "It froze over!"
    if "heaven" in location.lower(): return "It smells like oranges here"
    return "Quite pleasant"


async with Model("accounts/fireworks/models/kimi-k2p6", api_key=os.environ["FIREWORKS_API_KEY"], base_url="https://api.fireworks.ai/inference/v1/") as m:
# async with Model("Qwen/Qwen3.6-27B") as m:

    history = [
        Message("user", "What's the weaher in hell and heaven today? Answer the question using tool calls")
    ]
    print(history[-1])

    res = await m.generate(history=history, tools=[get_weather])
    history.append(res)
    print(history[-1])


    for tc in res.tool_calls:
        if tc.function.name == "get_weather":
            tool_res = get_weather(**tc.function.arguments)
            history.append(Message("tool", content=tool_res, tool_call_id=tc.id))
            print(history[-1])


    if len(res.tool_calls):
        res = await m.generate(history=history, tools=[get_weather])
        history.append(res)
        print(history[-1])

Message(role='user', content="What's the weaher in hell and heaven today? Answer the question using tool calls", tool_calls=[], tool_call_id=None)
Message(role='assistant', content='', tool_calls=[ToolCall(id='functions.get_weather:0', function=FunctionCall(name='get_weather', arguments={'location': 'Hell, Norway'}), type='function'), ToolCall(id='functions.get_weather:1', function=FunctionCall(name='get_weather', arguments={'location': 'Heaven, Michigan, USA'}), type='function')], tool_call_id=None)
get_weather(Hell, Norway)
Message(role='tool', content='It froze over!', tool_calls=[], tool_call_id='functions.get_weather:0')
get_weather(Heaven, Michigan, USA)
Message(role='tool', content='It smells like oranges here', tool_calls=[], tool_call_id='functions.get_weather:1')
Message(role='assistant', content='Based on the tool calls:\n\n**Hell, Norway**: It froze over! (A classic!)\n\n**Heaven, Michigan, USA**: It smells like oranges there today.\n\nSo it looks like Hell has frozen over,

I think this should be enough for our task. Let's get to WS and NEON message parsing


In [8]:
import websockets
from functools import partial

@dataclass
class NeonTransmission:
    type: Literal["challenge", "success", "error"]
    message: str | None = None


    @classmethod
    def from_recv(cls, data: dict):
        type = data["type"]
        if type == "challenge":
            fragments: list[dict] = data["message"]
            message = " ".join([f["word"] for f in sorted(fragments, key=lambda fr: fr["timestamp"]) ])
        elif type == "error":
            message = data["message"]
        elif type == "success":
            message = None
        else:
            raise ValueError
        return cls(type=data["type"], message=message)


async with websockets.connect("wss://neonhealth.software/agent-puzzle/challenge") as ws:
    res = json.loads(await ws.recv())
    tr = NeonTransmission.from_recv(res)

In [9]:
tr

NeonTransmission(type='challenge', message='Incoming vessel detected. If your pilot is an AI co-pilot built by an excellent software engineer, respond on frequency 2. All other vessels, respond on frequency 4.')

I think we can now create some tools, and put it all together. I'll try to keep it simple


In [ ]:
import quickjs # Python handles math slightly differently from JS.
from urllib.parse import quote

SYSTEM_PROMPT = """
You are an AI copilot, handle communications with NEON.

When confirming comms channel, identify as copilot.
Use one tool at a time, never batch them.
Pay attention. Make sure you are using the correct tool with the correct args.
Keep your responses concise.

You must solve each checkpoint by using the available tools.
Rules:
- Use send_digits for any raw keypad response that should be sent unchanged, including vessel authorization codes, hex strings, frequencies, and values followed by #.
- Use send_text for all text responses. If NEON gives minimum or maximum character counts, pass them so the text is padded or cropped before sending.
- The vessel authorization code is a hex string. Never evaluate or transform it; send it unchanged with send_digits.
- Use eval_and_send_digits only for arithmetic prompts where you must compute a result before sending keypad digits.
- Use wiki_summary_send_word for knowledge archive / Wikipedia summary responses so lookup and sending happen in one step.
- Use transmit_nth_word for transmission-verification prompts that ask for the Nth word from an earlier text you already sent.
- Be very concise and deliberate.

Use plain replies only for diagnostic messages - never for messagse intended for NEON, and only on errors.

When evaluating the JS expression, check for syntax erros before eval and correct them.
When NEON asks the response to be between X and Y characters, aim for the middle of the interval, even if you have to pad the response with water.
"""

from pathlib import Path

USER_PROMPT = f"""
Vessel code: 5155c6281426df78

Crew manifest:

{Path('docs/resume.md').read_text()}
"""

history = [
    Message("system", SYSTEM_PROMPT),
    Message("user", USER_PROMPT),
]

print(history[-2])
print(history[-1])

async with (websockets.connect("wss://neonhealth.software/agent-puzzle/challenge") as ws,
            Model("accounts/fireworks/models/kimi-k2p6", api_key=os.environ["FIREWORKS_API_KEY"], base_url="https://api.fireworks.ai/inference/v1/") as m):
            # Model("Qwen/Qwen3.5-9B", base_url="http://localhost:3456/v1/") as m): # Qwen 3.6 running locally! (but it's too dumb)

    # To keep things simple, I'll use `ws` from the closure.
    async def enter_digits(input: Annotated[str, "hex digits or #"]):
        """Send hex digits or # to NEON. Use this when requires to send digits"""

        print(f"NEON <= enter_digits('{input}')")

        await ws.send(json.dumps({ "type": "enter_digits", "digits": input }))

        return "Ok"
    
    async def speak_text(text: str, min_ch: int, max_ch: int):
        """Send a string to NEON, crop/pad the fix the min/max chars"""

        while len(text) < min_ch: text += " pad"
        text = text[:max_ch]

        print(f'NEON <= speak_text("{text}"), min={min_ch}, max={max_ch})')

        await ws.send(json.dumps({ "type": "speak_text", "text": text}))
                      
        return "OK"


    async def eval_send(expr: str, suffix: Annotated[str, "Append text, such as #, to the end of the result number"] ):
        """Evaluate a mathematical expression and transmit it to NEON"""

        ctx = quickjs.Context()

        res = str(ctx.eval(expr)) + suffix    

        await enter_digits(res)

        return f"Sent '{res}'"

    async def send_wiki_word(article: str, word_n: Annotated[int, "index starts at 1"]):
        """Find a word in a WikiPedia article and trnasmit it to NEON"""
        async with aiohttp.ClientSession() as wp:
            res = await wp.get(f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote(article)}",
                        headers={"User-Agent": "Space Balls https://alexey.work"} # WP requires a proper user-agent with a url
                )
        text: str = (await res.json())["extract"]
        words = text[:1000].split()
        print(words)
        word = ''.join(c for c in words[word_n-1] if c.isalnum) # Drop punctuation

        await speak_text(word, 1, 100)

        return f"Sent '{word}'"

    async def send_nth_word(text: str, word_n: int):
        """Find n-th word in 'text' and trnasmit it to NEON"""
        words = text.split()
        word = ''.join(c for c in words[word_n-1] if c.isalnum())

        await speak_text(word, 1, 100)

        return f"Sent '{word}'"

    tools: list[Callable] = [enter_digits, speak_text, eval_send, send_wiki_word, send_nth_word]



    while True: # Here NEON drives the loop. The LLM never has to make more than 1 tool call per prompt.
        res = json.loads(await ws.recv())
        tr = NeonTransmission.from_recv(res)

        print(f"NEON => {tr}")

        if tr.type != "challenge":
            break

        history.append(Message("user", f"NEON: {tr.message}"))
        print(history[-1])

        res = await m.generate(history, tools, "required")

        history.append(res)
        print(history[-1])
        
        for tc in res.tool_calls:
            for tool in tools:
                if tc.function.name == tool.__name__:
                    res = await tool(**tc.function.arguments)
                    history.append(Message("tool", res, tool_call_id=tc.id))
                    print(history[-1])

Message(role='system', content='\nYou are an AI copilot, handle communications with NEON.\n\nWhen confirming comms channel, identify as copilot.\nUse one tool at a time, never batch them.\nPay attention. Make sure you are using the correct tool with the correct args.\nKeep your responses concise.\n\nYou must solve each checkpoint by using the available tools.\nRules:\n- Use send_digits for any raw keypad response that should be sent unchanged, including vessel authorization codes, hex strings, frequencies, and values followed by #.\n- Use send_text for all text responses. If NEON gives minimum or maximum character counts, pass them so the text is padded or cropped before sending.\n- The vessel authorization code is a hex string. Never evaluate or transform it; send it unchanged with send_digits.\n- Use eval_and_send_digits only for arithmetic prompts where you must compute a result before sending keypad digits.\n- Use wiki_summary_send_word for knowledge archive / Wikipedia summary res